## 0: Autosave and Imports

In [ ]:
%autosave 60
%pip install --quiet -r requirements.txt

In [ ]:
import os
import yaml
import torch
import shutil
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

from utils.mri.data_loader import DataLoader
from utils.mri.data_converter import DataConverter
from utils.model_utils.bayes_search import init_yaml, run_bayes, select_bayes_champions
from utils.model_utils.train_eval import get_middle_slice_3D, get_metadata_features
from utils.model_utils.loss_functions import ssim_loss, l1_loss
from utils.analysis.data_analyser import DataAnalyser
from utils.metadata.health_data_utils import HealthDataLoader
from utils.metadata.fastsurfer_utils import FastSurferLoader, generate_all_csvs
from utils.metadata.combined_metadata_utils import CombinedMetadataUtils
from utils.metadata.stratified_splitter import StratifiedSplitter

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Data Setup
First prepare all the data used for training. All functions are safe to run multiple times, and will just skip preprocessing if it is already done.

In [ ]:
root_path       = 'data/'
metadata_root   = 'data/metadata'
healthdata_root = 'data/metadata/health_data'
processed_dir   = 'data/metadata/processed'
fastsurfer_root = 'data/metadata/hdd/sMRI_generated'
final_data_path = 'data/metadata/combined_metadata.csv'
overwrite_data  = False
ignore_training = True
random_seed     = 69

grid_path       = 'out/grid_search/grid_search.yaml'
bayes_path      = 'out/bayes_search/bayes_search.yaml'

data_loader     = DataLoader(root_path=root_path)
health_loader   = HealthDataLoader(root=metadata_root)
fs_loader       = FastSurferLoader(root=fastsurfer_root)
comb_loader     = CombinedMetadataUtils()

data_splitter   = StratifiedSplitter()

data_converter = DataConverter()
data_analyser  = DataAnalyser(root_path=root_path)

Convert data from SAV to CSV

In [ ]:
mapping_path = health_loader.sav_to_csv(overwrite=overwrite_data)

Clean Health Data

In [ ]:
cleaned_data_path = health_loader.generate_cleaned(overwrite=overwrite_data)

Load Health data, generating a normalized_health_data.csv

In [ ]:
health_data_path = health_loader.generate_normalized(overwrite=overwrite_data, health_data_path=cleaned_data_path, out_dir=processed_dir)

Generate CSV files containing all the stats from FreeSurfer

In [ ]:
generate_all_csvs(overwrite=overwrite_data, subjects_dir="data/metadata/hdd/full", output_dir=fastsurfer_root)

Load Fastsurfer data, generating a normalized_fastsurfer_data.csv

In [ ]:
fs_data_path = fs_loader.load_and_normalize(overwrite=overwrite_data, in_dir=fastsurfer_root, out_dir=processed_dir)

Combine data into a single csv table, which we can use going forward

In [ ]:
comb_loader.get_overlap(health_data_path=health_data_path, fastsurfer_data_path=fs_data_path);

In [ ]:
metadata_path = comb_loader.combine(overwrite=overwrite_data, health_data_path=health_data_path, fastsurfer_data_path=fs_data_path, output_path=metadata_root)
metadata_df = pd.read_csv(metadata_path)
metadata_df.head()

### Split dataset based on features

In [ ]:
train, val, test = data_splitter.split(metadata_df, id_col="hunt_id", save_path="data/metadata/splits.json", train_split=0.7, val_split=0.15, seed=random_seed)

# Print the number of samples in each split
print(f"Train samples: {len(train)}")
print(f"Validation samples: {len(val)}")
print(f"Test samples: {len(test)}")

## 2. Run Models

In [ ]:
if not ignore_training:
    init_yaml(yaml_file=bayes_path, epochs=4000, base_channels=16, restart_threshold=0.2, restart_check_epoch=500, max_total_runs=3)

In [ ]:
print(f"Train pairs: {len(train)}  Val pairs: {len(val)}")
train_pairs = [data_loader.get_pair_path_from_id(hunt_id) for hunt_id in train]
val_pairs   = [data_loader.get_pair_path_from_id(hunt_id) for hunt_id in val]

if not ignore_training:
    run_bayes(
        yaml_file=bayes_path,
        db_path='out/bayes_search/bayes_search.db',
        train_pairs=train_pairs,
        val_pairs=val_pairs,
        n_trials=10,
        metadata_root=metadata_root,
        out_dir='out/bayes_search',
        device=device,
    )

## 3. Fetch the Champions

Run these cells after all three models are trained (or loaded from disk).

In [ ]:
with open(bayes_path) as f:
    bayes_doc = yaml.safe_load(f)

completed = [e for e in bayes_doc['trials'] if e.get('results', {}).get('completed')]
print(f"Completed: {len(completed)} / {len(bayes_doc['trials'])} trials\n")

rows = []
for e in completed:
    r = e['results']
    rows.append({
        'id':             e['id'],
        'study':          e.get('study_name', '—'),
        'model_type':     e['model_type'],
        'film_type':      e.get('film_generator_type', '—'),
        'residual':       e.get('residual', '—'),
        'lr':             e['learning_rate'],
        'weight_decay':   e['weight_decay'],
        'scheduler':      e['lr_scheduler'],
        'mlp_hidden':     e.get('mlp_hidden', '—'),
        'best_val_loss':  r['best_val_loss'],
        'best_test_loss': r.get('best_test_loss', '—'),
        'trained_at':     r.get('trained_at', ''),
    })

df = pd.DataFrame(rows).sort_values('best_val_loss').reset_index(drop=True)
display(df)

In [ ]:
champions = select_bayes_champions(yaml_file=bayes_path, device=device, metadata_root=metadata_root)
print(len(champions))

## 5. Evaluate The Champions performance

### Qualitative

In [ ]:
CROP_AXES     = ((16, 10, 0), (17, 11, 17))

rng           = np.random.default_rng(random_seed)
sample_ids    = rng.choice(list(test), size=min(3, len(test)), replace=False).tolist()
sample_pairs  = [data_loader.get_pair_path_from_id(hid) for hid in sample_ids]
model_keys    = list(champions.keys())
n_models      = len(model_keys)
n_cols        = max(3, n_models)
DIFF_VMAX     = 0.3   # consistent heatmap upper bound across all diff panels
DIFF_CMAP     = 'hot'

# Slice positions along the W axis rendered in every panel (fractions of W).
# Each panel ends up as a horizontal triptych: ~33% | 50% | ~67% through the volume.
SLICE_FRACTIONS = (2/3, 1/2, 1/3)

clear_dir = True
save_dir = 'out/visualizations'
outlier_dir = 'out/visualizations/outliers'

In [ ]:
# Pick 3 random test subjects and render a qualitative comparison per subject.
# Each panel shows three slices stacked vertically (top 1/3, middle, bottom 2/3 of W).
# Emits two figures per subject:
#   fig 1: HUNT3 input | HUNT4 ground-truth | true residual |HUNT4 - HUNT3|
#   fig 2: predicted residual per champion, then a gap, then the true residual
#          so each model can be eyeballed against the ground-truth |Δ|
# Per-model panel titles: model name on top, SSIM and L1 (full 3D volume) below.
# Figure-level suptitle describes what the figure is showing.
if clear_dir and os.path.isdir(save_dir):
    shutil.rmtree(save_dir)
os.makedirs(save_dir, exist_ok=True)

# Per-panel target width (inches). Height is derived dynamically per subject to
# match the actual montage aspect ratio — otherwise imshow(aspect='equal')
# leaves a big vertical band of whitespace inside each axes box.
PANEL_W       = 2.5            # base panel width (inches)
TITLE_PAD     = 0.55           # inches reserved per panel for the 2-line axes title
SUPTITLE_INCH = 0.32           # inches reserved at top of figure for fig.suptitle
TIGHT_KW      = dict(pad=0.2, h_pad=0.15, w_pad=0.15)

# Suptitle text per row-kind.
ROW_LABELS = {
    'inputs':     'HUNT3 vs HUNT4',
    'pred_resid': 'Predicted residual',
}

def _needs_cond(key: str) -> bool:
    return key.startswith('film') or key == 'meta'

def _slice_montage(volume, fractions=SLICE_FRACTIONS):
    """Return a (len(fractions)*D, H) numpy array — slices at given fractional
    positions along the W axis stacked vertically (1xN layout), clamped to [0, 1]."""
    if isinstance(volume, torch.Tensor):
        t = volume.detach().cpu()
    else:
        t = torch.tensor(volume)
    _, _, _, _, W = t.shape
    parts = []
    for f in fractions:
        idx = max(0, min(W - 1, int(round(f * W))))
        parts.append(t[0, 0, :, :, idx])
    return torch.cat(parts, dim=0).clamp(0, 1).numpy()

def _panel_h(sample_slice):
    """Per-panel height (inches) sized to the montage aspect ratio + title pad."""
    sh, sw = sample_slice.shape
    return PANEL_W * sh / sw + TITLE_PAD

def _apply_suptitle(fig, fig_h_in, text):
    """Place a figure-level title in the SUPTITLE_INCH-tall reserved band at the
    top of the figure, then cap tight_layout to the area below it. fig_h_in is
    the figure height in inches (the same value passed to plt.subplots' figsize)."""
    frac = SUPTITLE_INCH / fig_h_in
    fig.suptitle(text, fontsize=11, fontweight='bold', y=1 - frac / 2)
    plt.tight_layout(rect=(0, 0, 1, 1 - frac), **TIGHT_KW)

def _draw_row0(ax_input, ax_gt, ax_diff, ax_hide, input_sl, gt_sl, mean_abs_resid_3d):
    """Row 1 — input | ground truth | |HUNT4 - HUNT3| (each is a 3-slice montage)."""
    ax_input.imshow(input_sl, cmap='gray', vmin=0, vmax=1)
    ax_input.set_title('HUNT3 (input)', fontsize=10, fontweight='bold')
    ax_gt.imshow(gt_sl, cmap='gray', vmin=0, vmax=1)
    ax_gt.set_title('HUNT4 (ground truth)', fontsize=10, fontweight='bold')
    ax_diff.imshow(np.abs(gt_sl - input_sl), cmap=DIFF_CMAP, vmin=0, vmax=DIFF_VMAX)
    ax_diff.set_title(
        f'True residual\n3D mean |Δ| = {mean_abs_resid_3d:.4f}',
        fontsize=10, fontweight='bold',
    )
    for ax in (ax_input, ax_gt, ax_diff):
        ax.axis('off')
    for ax in ax_hide:
        ax.set_visible(False)

def _draw_model_row(axes_row, kind, input_sl, gt_sl, preds):
    """Draw one row of per-model panels. kind in {'pred_resid', 'pred', 'error'}.
    Each panel title is 2-line: model name (top) and 3D SSIM/L1 (below)."""
    for c, key in enumerate(model_keys):
        ax = axes_row[c]
        pred_sl = preds[key]['slice']
        if kind == 'pred_resid':
            ax.imshow(np.abs(pred_sl - input_sl), cmap=DIFF_CMAP, vmin=0, vmax=DIFF_VMAX)
        elif kind == 'pred':
            ax.imshow(pred_sl, cmap='gray', vmin=0, vmax=1)
        elif kind == 'error':
            ax.imshow(np.abs(pred_sl - gt_sl), cmap=DIFF_CMAP, vmin=0, vmax=DIFF_VMAX)
        ax.set_title(
            f'{key}\nSSIM={preds[key]["ssim"]:.4f}   L1={preds[key]["l1"]:.4f}',
            fontsize=9,
        )
        ax.axis('off')
    for c in range(n_models, len(axes_row)):
        axes_row[c].set_visible(False)

def _save_and_show(fig, filename, dir=save_dir):
    fig.savefig(os.path.join(dir, filename), dpi=150, bbox_inches='tight')
    plt.show()

for hid, (x_path, y_path) in zip(sample_ids, sample_pairs):
    # Crop once per subject; reuse across models. Each model lives on its own
    # device (std splits across GPUs), so move inputs per-model.
    x_full = data_converter.load_path_as_tensor(x_path, device)
    y_full = data_converter.load_path_as_tensor(y_path, device)
    x_crop = data_converter.get_volume_with_3d_change(x_full, CROP_AXES, remove_mode=True)
    y_crop = data_converter.get_volume_with_3d_change(y_full, CROP_AXES, remove_mode=True)
    input_sl = _slice_montage(x_crop)
    gt_sl    = _slice_montage(y_crop)
    # 3D mean |HUNT4 - HUNT3| over the full crop — gives a per-subject "how much
    # did this person actually change" number, independent of which slices we render.
    mean_abs_resid_3d = float(torch.mean(torch.abs(y_crop - x_crop)).item())
    panel_h = _panel_h(input_sl)

    preds = {}
    with torch.no_grad():
        for key, model in champions.items():
            data_device = next(model.parameters()).device
            x_d = x_crop.to(data_device)
            y_d = y_crop.to(data_device)
            if _needs_cond(key):
                cond = get_metadata_features(comb_loader, x_path).unsqueeze(0).to(data_device)
                out  = model(x_d, cond)
            else:
                out  = model(x_d)
            pred = out[0] if isinstance(out, (tuple, list)) else out
            preds[key] = dict(
                slice=_slice_montage(pred),
                ssim=float(ssim_loss(pred, y_d).item()),
                l1  =float(l1_loss(pred, y_d).item()),
            )
            del x_d, y_d, out, pred
    del x_full, y_full, x_crop, y_crop
    torch.cuda.empty_cache()

    print(f'=== Subject {hid} ===')

    # Figure 1 — fixed 3-panel figure (input | gt | true residual)
    fig_h = panel_h + SUPTITLE_INCH
    fig, axes = plt.subplots(1, 3, figsize=(PANEL_W * 3, fig_h))
    _draw_row0(axes[0], axes[1], axes[2], [], input_sl, gt_sl, mean_abs_resid_3d)
    _apply_suptitle(fig, fig_h, ROW_LABELS['inputs'])
    _save_and_show(fig, f'subject_{hid}_inputs.png')

    # Figure 2 — predicted residuals per champion, then a small gap, then the
    # true residual so each model's prediction can be eyeballed against the
    # ground-truth |Δ|.
    fig_h = panel_h + SUPTITLE_INCH
    spacer_ratio = 0.1
    fig_w = PANEL_W * (n_models + 1 + spacer_ratio)
    fig, axes = plt.subplots(
        1, n_models + 2,
        figsize=(fig_w, fig_h),
        gridspec_kw={'width_ratios': [1] * n_models + [spacer_ratio, 1]},
        squeeze=False,
    )
    _draw_model_row(axes[0, :n_models], 'pred_resid', input_sl, gt_sl, preds)
    axes[0, n_models].set_visible(False)
    ax_true = axes[0, n_models + 1]
    ax_true.imshow(np.abs(gt_sl - input_sl), cmap=DIFF_CMAP, vmin=0, vmax=DIFF_VMAX)
    ax_true.set_title(
        f'True residual\n3D mean |Δ| = {mean_abs_resid_3d:.4f}',
        fontsize=9,
    )
    ax_true.axis('off')
    _apply_suptitle(fig, fig_h, ROW_LABELS['pred_resid'])
    _save_and_show(fig, f'subject_{hid}_predicted_residuals.png')


#### Outliers

In [ ]:
# Outlier variant of the qualitative comparison: emits two figures per subject —
# the HUNT3/HUNT4/true-residual triptych, then the predicted residuals across
# champions with the true residual appended (after a small gap) for side-by-side
# evaluation. Reuses _slice_montage, _panel_h, _apply_suptitle, _draw_row0,
# _draw_model_row, _save_and_show, PANEL_W, SUPTITLE_INCH, DIFF_CMAP, DIFF_VMAX,
# ROW_LABELS from above.
# Subject IDs to render as qualitative outlier examples. Withheld from the
# public release for privacy; populate with IDs from your own test split.
outlier_ids   = []
outlier_pairs = [data_loader.get_pair_path_from_id(hid) for hid in outlier_ids]

# clear_dir is honored once per cell; if it was already cleared by the cell
# above this is a no-op because outlier_dir was just recreated.
if clear_dir and os.path.isdir(outlier_dir):
    shutil.rmtree(outlier_dir)
os.makedirs(outlier_dir, exist_ok=True)

for hid, (x_path, y_path) in zip(outlier_ids, outlier_pairs):
    x_full = data_converter.load_path_as_tensor(x_path, device)
    y_full = data_converter.load_path_as_tensor(y_path, device)
    x_crop = data_converter.get_volume_with_3d_change(x_full, CROP_AXES, remove_mode=True)
    y_crop = data_converter.get_volume_with_3d_change(y_full, CROP_AXES, remove_mode=True)
    input_sl = _slice_montage(x_crop)
    gt_sl    = _slice_montage(y_crop)
    # 3D mean |HUNT4 - HUNT3| — same metric as the random-sample cell.
    mean_abs_resid_3d = float(torch.mean(torch.abs(y_crop - x_crop)).item())
    panel_h = _panel_h(input_sl)

    preds = {}
    with torch.no_grad():
        for key, model in champions.items():
            data_device = next(model.parameters()).device
            x_d = x_crop.to(data_device)
            y_d = y_crop.to(data_device)
            if _needs_cond(key):
                cond = get_metadata_features(comb_loader, x_path).unsqueeze(0).to(data_device)
                out  = model(x_d, cond)
            else:
                out  = model(x_d)
            pred = out[0] if isinstance(out, (tuple, list)) else out
            preds[key] = dict(
                slice=_slice_montage(pred),
                ssim=float(ssim_loss(pred, y_d).item()),
                l1  =float(l1_loss(pred, y_d).item()),
            )
            del x_d, y_d, out, pred
    del x_full, y_full, x_crop, y_crop
    torch.cuda.empty_cache()

    print(f'=== Outlier {hid} ===')

    fig_h = panel_h + SUPTITLE_INCH
    fig, axes = plt.subplots(1, 3, figsize=(PANEL_W * 3, fig_h))
    _draw_row0(axes[0], axes[1], axes[2], [], input_sl, gt_sl, mean_abs_resid_3d)
    _apply_suptitle(fig, fig_h, ROW_LABELS['inputs'])
    _save_and_show(fig, f'{hid}_inputs.png', dir=outlier_dir)

    # Predicted residuals per champion, then a small gap, then the true residual
    # so each model's prediction can be eyeballed against the ground-truth |Δ|.
    fig_h = panel_h + SUPTITLE_INCH
    spacer_ratio = 0.1
    fig_w = PANEL_W * (n_models + 1 + spacer_ratio)
    fig, axes = plt.subplots(
        1, n_models + 2,
        figsize=(fig_w, fig_h),
        gridspec_kw={'width_ratios': [1] * n_models + [spacer_ratio, 1]},
        squeeze=False,
    )
    _draw_model_row(axes[0, :n_models], 'pred_resid', input_sl, gt_sl, preds)
    axes[0, n_models].set_visible(False)
    ax_true = axes[0, n_models + 1]
    ax_true.imshow(np.abs(gt_sl - input_sl), cmap=DIFF_CMAP, vmin=0, vmax=DIFF_VMAX)
    ax_true.set_title(
        f'True residual\n3D mean |Δ| = {mean_abs_resid_3d:.4f}',
        fontsize=9,
    )
    ax_true.axis('off')
    _apply_suptitle(fig, fig_h, ROW_LABELS['pred_resid'])
    _save_and_show(fig, f'{hid}_predicted_residuals.png', dir=outlier_dir)


### Quantitative

In [ ]:
stop

In [ ]:
def evaluate_model(model, pairs, conditional=False):
    model.eval()
    data_device = next(model.parameters()).device
    ssim_vals, l1_vals = [], []

    with torch.no_grad():
        for x_path, y_path in tqdm(pairs, desc='Evaluating'):
            x_full = data_converter.load_path_as_tensor(x_path, data_device)
            y_full = data_converter.load_path_as_tensor(y_path, data_device)
            x = data_converter.get_volume_with_3d_change(x_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True).to(data_device)
            y = data_converter.get_volume_with_3d_change(y_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True).to(data_device)

            if conditional:
                cond = get_metadata_features(cond_loader, x_path).unsqueeze(0).to(data_device)
                out  = model(x, cond)
            else:
                out = model(x)

            y_hat = out[0] if isinstance(out, (tuple, list)) else out
            ssim_vals.append(ssim_loss(y_hat, y).item())
            l1_vals.append(l1_loss(y_hat, y).item())
            del x_full, y_full, x, y, out, y_hat

    return dict(
        ssim_mean=float(np.mean(ssim_vals)),
        ssim_std=float(np.std(ssim_vals)),
        l1_mean=float(np.mean(l1_vals)),
        l1_std=float(np.std(l1_vals)),
    )

test_results = {}
for key, model in champions.items():
    print(f'Evaluating {key}...')
    test_results[key] = evaluate_model(model, test_pairs, conditional=key.startswith('film'))


### Estimate DICE

In [ ]:
import os, sys, subprocess
import nibabel as nib

FASTSURFER_DIR = os.path.join(os.path.expanduser("~"), "FastSurfer")
GT_SEG_DIR     = "out/hunt4_gt/segmentations"
THRESHOLD      = 1e-3

os.makedirs(GT_SEG_DIR, exist_ok=True)

GM_LABELS = set([
    *range(1000, 1036), *range(2000, 2036),
    10, 11, 12, 13, 17, 18, 26, 28,
    49, 50, 51, 52, 53, 54, 58, 60,
])
WM_LABELS  = set([2, 41, 7, 46, 77, 85, 251, 252, 253, 254, 255])
CSF_LABELS = set([4, 5, 14, 15, 24, 31, 43, 44, 63, 72])

def to_binary_mask(seg_vol, label_set):
    mask = np.zeros(seg_vol.shape, dtype=bool)
    for label in label_set:
        mask |= (seg_vol == label)
    return mask

def dice_coefficient(mask_pred, mask_gt):
    intersection = np.logical_and(mask_pred, mask_gt).sum()
    total = mask_pred.sum() + mask_gt.sum()
    return float('nan') if total == 0 else 2.0 * float(intersection) / float(total)

def volumetric_similarity(mask_pred, mask_gt):
    v_pred = float(mask_pred.sum())
    v_gt   = float(mask_gt.sum())
    denom  = v_pred + v_gt
    return float('nan') if denom == 0 else 1.0 - abs(v_pred - v_gt) / denom

def run_fastsurfer_seg(input_path, output_path):
    abs_out = os.path.abspath(output_path)
    sid     = os.path.basename(os.path.dirname(abs_out))
    sd      = os.path.dirname(os.path.dirname(abs_out))
    os.makedirs(os.path.dirname(abs_out), exist_ok=True)
    cmd = [
        sys.executable,
        os.path.join(FASTSURFER_DIR, "FastSurferCNN", "run_prediction.py"),
        "--t1",              os.path.abspath(input_path),
        "--asegdkt_segfile", abs_out,
        "--sd",              sd, "--sid", sid,
        "--device",          "cuda",
        "--batch_size",      "1",
    ]
    env = {**os.environ, "PYTHONPATH": FASTSURFER_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")}
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"[ERROR] FastSurfer failed on {input_path}\n{result.stderr[-800:]}")
        return False
    return True

# ── Segment GT once (reuses evaluation.ipynb's cache) ──────────────────────
n_skipped = 0
for x_path, y_path in tqdm(test_pairs, desc="GT segmentation"):
    patient_id  = os.path.basename(os.path.dirname(x_path))
    gt_seg_path = os.path.join(GT_SEG_DIR, patient_id, "gt_seg.nii.gz")
    if os.path.isfile(gt_seg_path):
        n_skipped += 1
        continue
    run_fastsurfer_seg(y_path, gt_seg_path)
print(f"GT segmentation done  ({n_skipped} already cached)")

# ── Inference → NIfTI → FastSurfer → Dice per champion ────────────────────
for key, model in champions.items():
    data_device = next(model.parameters()).device
    pred_dir    = f"out/grid_champions/{key}/predictions"
    seg_dir     = f"out/grid_champions/{key}/segmentations"
    os.makedirs(pred_dir, exist_ok=True)

    # --- save NIfTI predictions (cached) ---
    n_skip = 0
    for x_path, y_path in tqdm(test_pairs, desc=f"Inference {key}"):
        patient_id = os.path.basename(os.path.dirname(x_path))
        pred_path  = os.path.join(pred_dir, f"{patient_id}_pred.nii.gz")
        if os.path.isfile(pred_path):
            n_skip += 1
            continue
        with torch.no_grad():
            x_t = data_converter.load_path_as_tensor(x_path, data_device)
            x_c = data_converter.get_volume_with_3d_change(x_t, ((16, 10, 0), (17, 11, 17)), remove_mode=True)
            if key.startswith('film'):
                cond = get_metadata_features(cond_loader, x_path).unsqueeze(0).to(data_device)
                out  = model(x_c, cond)
            else:
                out = model(x_c)
            pred = out[0] if isinstance(out, (tuple, list)) else out
            pred[pred.abs() < THRESHOLD] = 0.0
            pred_full = data_converter.get_volume_with_3d_change(
                pred.cpu(), ((16, 10, 0), (17, 11, 17)), remove_mode=False)
        ref = nib.load(y_path)
        nib.save(nib.Nifti1Image(pred_full.squeeze().numpy().astype(np.float32),
                                 ref.affine, ref.header), pred_path)
        del x_t, x_c, out, pred, pred_full
        torch.cuda.empty_cache()
    print(f"  {key}: inference done  ({n_skip} cached)")

    # --- FastSurfer on predictions (cached) ---
    n_skip = 0
    for x_path, _ in tqdm(test_pairs, desc=f"Segmenting {key}"):
        patient_id = os.path.basename(os.path.dirname(x_path))
        pred_path  = os.path.join(pred_dir,  f"{patient_id}_pred.nii.gz")
        seg_path   = os.path.join(seg_dir,   patient_id, "pred_seg.nii.gz")
        if os.path.isfile(seg_path):
            n_skip += 1
            continue
        run_fastsurfer_seg(pred_path, seg_path)
    print(f"  {key}: segmentation done  ({n_skip} cached)")

    # --- Dice scores ---
    dice_rows = []
    for x_path, _ in test_pairs:
        patient_id    = os.path.basename(os.path.dirname(x_path))
        pred_seg_path = os.path.join(seg_dir,    patient_id, "pred_seg.nii.gz")
        gt_seg_path   = os.path.join(GT_SEG_DIR, patient_id, "gt_seg.nii.gz")
        if not os.path.isfile(pred_seg_path) or not os.path.isfile(gt_seg_path):
            print(f"  [WARN] missing seg for {patient_id}")
            continue
        pred_seg = np.round(nib.load(pred_seg_path).get_fdata()).astype(np.int32)
        gt_seg   = np.round(nib.load(gt_seg_path).get_fdata()).astype(np.int32)
        gm_pred  = to_binary_mask(pred_seg, GM_LABELS)
        gm_gt    = to_binary_mask(gt_seg,   GM_LABELS)
        wm_pred  = to_binary_mask(pred_seg, WM_LABELS)
        wm_gt    = to_binary_mask(gt_seg,   WM_LABELS)
        csf_pred = to_binary_mask(pred_seg, CSF_LABELS)
        csf_gt   = to_binary_mask(gt_seg,   CSF_LABELS)
        dice_rows.append({
            "dice_GM":  dice_coefficient(gm_pred,  gm_gt),
            "dice_WM":  dice_coefficient(wm_pred,  wm_gt),
            "dice_CSF": dice_coefficient(csf_pred, csf_gt),
            "vs_GM":    volumetric_similarity(gm_pred,  gm_gt),
            "vs_WM":    volumetric_similarity(wm_pred,  wm_gt),
            "vs_CSF":   volumetric_similarity(csf_pred, csf_gt),
        })
    df_dice = pd.DataFrame(dice_rows)
    test_results[key].update({
        "dice_gm_mean":  float(df_dice.dice_GM.mean()),
        "dice_gm_std":   float(df_dice.dice_GM.std()),
        "dice_wm_mean":  float(df_dice.dice_WM.mean()),
        "dice_wm_std":   float(df_dice.dice_WM.std()),
        "dice_csf_mean": float(df_dice.dice_CSF.mean()),
        "dice_csf_std":  float(df_dice.dice_CSF.std()),
        "vs_gm_mean":    float(df_dice.vs_GM.mean()),
        "vs_gm_std":     float(df_dice.vs_GM.std()),
        "vs_wm_mean":    float(df_dice.vs_WM.mean()),
        "vs_wm_std":     float(df_dice.vs_WM.std()),
        "vs_csf_mean":   float(df_dice.vs_CSF.mean()),
        "vs_csf_std":    float(df_dice.vs_CSF.std()),
    })
    print(f"  {key}: Dice  GM={df_dice.dice_GM.mean():.4f}  WM={df_dice.dice_WM.mean():.4f}  CSF={df_dice.dice_CSF.mean():.4f}")
    print(f"  {key}: VS    GM={df_dice.vs_GM.mean():.4f}   WM={df_dice.vs_WM.mean():.4f}   CSF={df_dice.vs_CSF.mean():.4f}")

In [ ]:
header = '{:<20}  {:>10}  {:>10}  {:>8}  {:>8}  {:>8}  {:>8}  {:>8}  {:>8}'.format(
    'Model', 'SSIM Loss', 'L1 Loss', 'Dice GM', 'Dice WM', 'Dice CSF', 'VS GM', 'VS WM', 'VS CSF')
print(header)
print('-' * len(header))
for name, r in test_results.items():
    dice_gm  = f"{r['dice_gm_mean']:.4f}\xb1{r['dice_gm_std']:.4f}"   if 'dice_gm_mean'  in r else '—'
    dice_wm  = f"{r['dice_wm_mean']:.4f}\xb1{r['dice_wm_std']:.4f}"   if 'dice_wm_mean'  in r else '—'
    dice_csf = f"{r['dice_csf_mean']:.4f}\xb1{r['dice_csf_std']:.4f}" if 'dice_csf_mean' in r else '—'
    vs_gm    = f"{r['vs_gm_mean']:.4f}\xb1{r['vs_gm_std']:.4f}"       if 'vs_gm_mean'    in r else '—'
    vs_wm    = f"{r['vs_wm_mean']:.4f}\xb1{r['vs_wm_std']:.4f}"       if 'vs_wm_mean'    in r else '—'
    vs_csf   = f"{r['vs_csf_mean']:.4f}\xb1{r['vs_csf_std']:.4f}"     if 'vs_csf_mean'   in r else '—'
    print('{:<20}  {:>10.5f}  {:>10.5f}  {:>8}  {:>8}  {:>8}  {:>8}  {:>8}  {:>8}'.format(
        name, r['ssim_mean'], r['l1_mean'], dice_gm, dice_wm, dice_csf, vs_gm, vs_wm, vs_csf))


In [ ]:
names  = list(test_results.keys())
colors = ['#4C72B0', '#DD8452', '#55A868'][:len(names)]
has_dice = all('dice_gm_mean' in test_results[n] for n in names)
has_vs   = all('vs_gm_mean'   in test_results[n] for n in names)

n_rows = 1 + (1 if has_vs else 0)
n_cols = 5 if has_dice else 2
fig, axes_grid = plt.subplots(n_rows, n_cols, figsize=(22, 5 * n_rows))
axes = axes_grid.flatten() if n_rows > 1 else list(axes_grid) if has_dice else list(axes_grid)

def _bar(ax, title, ylabel, values, errs, ylim=None):
    ax.bar(names, values, yerr=errs, color=colors, capsize=6, width=0.5, edgecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(ylabel)
    if ylim: ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.3)

_bar(axes[0], 'SSIM Loss  (lower = better)', '1 - SSIM',
     [test_results[n]['ssim_mean'] for n in names],
     [test_results[n]['ssim_std']  for n in names], ylim=(0, None))
_bar(axes[1], 'L1 Loss  (lower = better)', 'MAE',
     [test_results[n]['l1_mean'] for n in names],
     [test_results[n]['l1_std']  for n in names], ylim=(0, None))

if has_dice:
    for ax, title, km, ks in [
        (axes[2], 'Dice GM  (higher = better)',  'dice_gm_mean',  'dice_gm_std'),
        (axes[3], 'Dice WM  (higher = better)',  'dice_wm_mean',  'dice_wm_std'),
        (axes[4], 'Dice CSF (higher = better)',  'dice_csf_mean', 'dice_csf_std'),
    ]:
        _bar(ax, title, 'Dice', [test_results[n][km] for n in names],
             [test_results[n][ks] for n in names], ylim=(0, 1))

if has_vs:
    offset = n_cols  # second row starts here
    for ax, title, km, ks in [
        (axes[offset],   'VS GM  (higher = better)',  'vs_gm_mean',  'vs_gm_std'),
        (axes[offset+1], 'VS WM  (higher = better)',  'vs_wm_mean',  'vs_wm_std'),
        (axes[offset+2], 'VS CSF (higher = better)',  'vs_csf_mean', 'vs_csf_std'),
    ]:
        _bar(ax, title, 'VS', [test_results[n][km] for n in names],
             [test_results[n][ks] for n in names], ylim=(0, 1))
    # hide unused axes in second row
    for ax in axes[offset+3:]:
        ax.set_visible(False)

plt.tight_layout()
plt.show()


### Visual Reconstruction Comparison

In [ ]:
x_path, y_path = test_pairs[0]

with torch.no_grad():
    recons = {}
    for key, model in champions.items():
        data_device = next(model.parameters()).device
        x_full = data_converter.load_path_as_tensor(x_path, data_device)
        y_full = data_converter.load_path_as_tensor(y_path, data_device)
        x = data_converter.get_volume_with_3d_change(x_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True)
        y = data_converter.get_volume_with_3d_change(y_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True)

        if key.startswith('film'):
            cond = get_metadata_features(cond_loader, x_path).unsqueeze(0).to(data_device)
            out = model(x, cond)
        else:
            out = model(x)
        recons[key] = out[0] if isinstance(out, (tuple, list)) else out

x_sl = get_middle_slice_3D(x)
y_sl = get_middle_slice_3D(y)

panels = [('HUNT3 (input)', x_sl), ('HUNT4 (target)', y_sl)] + \
         [(key, get_middle_slice_3D(recons[key])) for key in recons]

n_cols = len(panels)
fig, axes = plt.subplots(2, n_cols, figsize=(4 * n_cols, 8))

for col, (title, sl) in enumerate(panels):
    axes[0, col].imshow(sl, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(title, fontsize=11, fontweight='bold')
    axes[0, col].axis('off')

    if title not in ('HUNT3 (input)', 'HUNT4 (target)'):
        diff = np.abs(sl - y_sl)
        axes[1, col].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
        axes[1, col].set_title('Diff  MAE={:.4f}'.format(np.mean(diff)), fontsize=10)
        axes[1, col].axis('off')
    else:
        axes[1, col].set_visible(False)

fig.suptitle('Visual Comparison — First Test Subject', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
